# Extended Classifications - All Firm Categories

This notebook creates all required dummy variables for the assignment:
- **Years covered**: 2020, 2021, 2022, 2023, 2024
- **Categories**: Scalers, HGFs, Consistent HGFs, Consistent Hypergrowers, Gazelles, Mature HGFs, Scaleups, Superstars
- **Total variables**: 50 (5 years × 10 categories including growth and AAGR)

## Key Rules
- **Rule 1**: If input employee value is unavailable, output should be unavailable ("n.a.")
- **Rule 4**: Size threshold of 10 employees at beginning of 3-year period
- Rolling 3-year windows for each classification year

In [1]:
import pandas as pd
import numpy as np
import datetime

# Load the data with growth variables
df = pd.read_pickle('../data/processed/austria_with_growth.pkl')
print(f"Loaded data: {df.shape}")
print(f"Available columns: {df.columns.tolist()[:10]}...")

Loaded data: (46085, 31)
Available columns: ['company_name', 'country_code', 'city', 'nace_code', 'bvd_id', 'nace_section', 'region_raw', 'status', 'incorporation_date', 'emp_2024_raw']...


## 1. Add Growth and AAGR for 2020, 2021, 2022, 2023

In [2]:
# For classification ending in 2020: use years 2018, 2019, 2020
df['growth_2020'] = (df['emp_2020_num'] - df['emp_2019_num']) / df['emp_2019_num']
df['growth_2019'] = (df['emp_2019_num'] - df['emp_2018_num']) / df['emp_2018_num']
df['growth_2018'] = (df['emp_2018_num'] - df['emp_2017_num']) / df['emp_2017_num']

# For classification ending in 2021: use years 2019, 2020, 2021
df['growth_2021'] = (df['emp_2021_num'] - df['emp_2020_num']) / df['emp_2020_num']

# For classification ending in 2022: use years 2020, 2021, 2022 (already have growth_2022)

# For classification ending in 2023: use years 2021, 2022, 2023 (already have growth_2023)

# For classification ending in 2024: use years 2022, 2023, 2024 (already have growth_2024)

# Mark as 'n.a.' where needed
growth_cols_new = ['growth_2020', 'growth_2019', 'growth_2018', 'growth_2021']
for col in growth_cols_new:
    df[col] = df[col].fillna('n.a.')

print("Growth variables added for all years")

Growth variables added for all years


In [3]:
# Calculate AAGR for each classification year
# aagr_2020 = (emp_2020 / emp_2017)^(1/3) - 1
# aagr_2021 = (emp_2021 / emp_2018)^(1/3) - 1
# aagr_2022 = (emp_2022 / emp_2019)^(1/3) - 1 (we need to recalc, currently it's emp_2024/emp_2021)
# aagr_2023 = (emp_2023 / emp_2020)^(1/3) - 1
# aagr_2024 = (emp_2024 / emp_2021)^(1/3) - 1 (already have)

def calculate_aagr(df, end_year, start_year):
    """Calculate AAGR for a given period"""
    emp_end = df[f'emp_{end_year}_num']
    emp_start = df[f'emp_{start_year}_num']
    
    # Calculate ratio
    ratio = emp_end / emp_start
    
    # Calculate AAGR: (ratio)^(1/3) - 1, then * 100
    aagr = (ratio ** (1/3) - 1) * 100
    
    # Create column as object type
    aagr_col = aagr.astype(object)
    
    # Mark as 'n.a.' if:
    # - Either endpoint missing
    # - Start year employees < 10
    mask_na = (
        emp_end.isna() |
        emp_start.isna() |
        (emp_start < 10)
    )
    aagr_col[mask_na] = 'n.a.'
    
    return aagr_col

# Calculate AAGR for each classification year
df['aagr_2020'] = calculate_aagr(df, 2020, 2017)
df['aagr_2021'] = calculate_aagr(df, 2021, 2018)
df['aagr_2022'] = calculate_aagr(df, 2022, 2019)
df['aagr_2023'] = calculate_aagr(df, 2023, 2020)
# aagr_2024 already exists

print("AAGR variables added for all classification years")
print(f"Total columns now: {len(df.columns)}")

AAGR variables added for all classification years
Total columns now: 39


## 2. Create Firm Category Classifications

For each year (2020-2024):
- **Scalers**: AAGR > 10% AND emp_start >= 10
- **HGFs**: AAGR > 20% AND emp_start >= 10
- **Consistent HGFs**: HGF AND growth > 20% in >= 2 of 3 years
- **Consistent Hypergrowers**: HGF AND growth > 40% in >= 2 of 3 years
- **Gazelles**: Consistent HGF AND company age <= 10 years
- **Mature HGFs**: Consistent HGF AND company age > 10 years
- **Scaleups**: Consistent Hypergrower AND company age <= 10 years
- **Superstars**: Consistent Hypergrower AND company age > 10 years

In [4]:
def classify_firms(df, end_year):
    """Create all firm category classifications for a given end year"""
    
    start_year = end_year - 3
    emp_start = df[f'emp_{start_year}_num']
    emp_end = df[f'emp_{end_year}_num']
    aagr_col = df[f'aagr_{end_year}']
    
    # Get growth rates for the 3 years
    growth_year_1 = df[f'growth_{start_year + 1}']  # First year growth
    growth_year_2 = df[f'growth_{start_year + 2}']  # Second year growth
    growth_year_3 = df[f'growth_{end_year}']        # Third year growth
    
    # Convert growth to numeric for comparison
    def to_numeric(col):
        result = pd.to_numeric(col, errors='coerce')
        return result
    
    g1_num = to_numeric(growth_year_1)
    g2_num = to_numeric(growth_year_2)
    g3_num = to_numeric(growth_year_3)
    aagr_num = to_numeric(aagr_col)
    
    # Classifiable = has valid AAGR value
    is_classifiable = aagr_num.notna()
    
    # Calculate company age at start of period
    founded_year = df['founded_year']
    company_age = start_year - founded_year
    is_young = company_age <= 10
    is_mature = company_age > 10
    
    # Initialize classification columns as 'n.a.'
    prefix = f'{end_year}_'
    
    # Scalers: AAGR > 10%
    df[f'{prefix}scaler'] = 'n.a.'
    df.loc[is_classifiable & (aagr_num > 10), f'{prefix}scaler'] = 1
    df.loc[is_classifiable & (aagr_num <= 10), f'{prefix}scaler'] = 0
    
    # HGFs: AAGR > 20%
    is_hgf = is_classifiable & (aagr_num > 20)
    df[f'{prefix}hgf'] = 'n.a.'
    df.loc[is_classifiable & (aagr_num > 20), f'{prefix}hgf'] = 1
    df.loc[is_classifiable & (aagr_num <= 20), f'{prefix}hgf'] = 0
    
    # Count years with growth > 20%
    years_gt_20 = ((g1_num > 20).astype(int) + 
                   (g2_num > 20).astype(int) + 
                   (g3_num > 20).astype(int))
    
    # Consistent HGFs: HGF AND growth > 20% in >= 2 of 3 years
    is_consistent_hgf = is_hgf & (years_gt_20 >= 2)
    df[f'{prefix}consistent_hgf'] = 'n.a.'
    df.loc[is_consistent_hgf, f'{prefix}consistent_hgf'] = 1
    df.loc[is_classifiable & ~is_consistent_hgf, f'{prefix}consistent_hgf'] = 0
    
    # Count years with growth > 40%
    years_gt_40 = ((g1_num > 40).astype(int) + 
                   (g2_num > 40).astype(int) + 
                   (g3_num > 40).astype(int))
    
    # Consistent Hypergrowers: HGF AND growth > 40% in >= 2 of 3 years
    is_hypergrower = is_hgf & (years_gt_40 >= 2)
    df[f'{prefix}consistent_hypergrower'] = 'n.a.'
    df.loc[is_hypergrower, f'{prefix}consistent_hypergrower'] = 1
    df.loc[is_classifiable & ~is_hypergrower, f'{prefix}consistent_hypergrower'] = 0
    
    # Gazelles: Consistent HGF AND age <= 10
    is_gazelle = is_consistent_hgf & is_young
    df[f'{prefix}gazelle'] = 'n.a.'
    df.loc[is_gazelle, f'{prefix}gazelle'] = 1
    df.loc[is_consistent_hgf & ~is_young, f'{prefix}gazelle'] = 0
    
    # Mature HGFs: Consistent HGF AND age > 10
    is_mature_hgf = is_consistent_hgf & is_mature
    df[f'{prefix}mature_hgf'] = 'n.a.'
    df.loc[is_mature_hgf, f'{prefix}mature_hgf'] = 1
    df.loc[is_consistent_hgf & ~is_mature, f'{prefix}mature_hgf'] = 0
    
    # Scaleups: Consistent Hypergrower AND age <= 10
    is_scaleup = is_hypergrower & is_young
    df[f'{prefix}scaleup'] = 'n.a.'
    df.loc[is_scaleup, f'{prefix}scaleup'] = 1
    df.loc[is_hypergrower & ~is_young, f'{prefix}scaleup'] = 0
    
    # Superstars: Consistent Hypergrower AND age > 10
    is_superstar = is_hypergrower & is_mature
    df[f'{prefix}superstar'] = 'n.a.'
    df.loc[is_superstar, f'{prefix}superstar'] = 1
    df.loc[is_hypergrower & ~is_mature, f'{prefix}superstar'] = 0
    
    return df

# Apply classification for each year
for year in [2020, 2021, 2022, 2023, 2024]:
    print(f"Classifying firms for year {year}...")
    df = classify_firms(df, year)

print(f"\nClassifications complete!")
print(f"Total columns: {len(df.columns)}")

Classifying firms for year 2020...
Classifying firms for year 2021...
Classifying firms for year 2022...
Classifying firms for year 2023...
Classifying firms for year 2024...

Classifications complete!
Total columns: 79


## 3. Summary Statistics

In [5]:
# Check classification statistics
print("\n=== Classification Summary ===")
for year in [2020, 2021, 2022, 2023, 2024]:
    prefix = f'{year}_'
    print(f"\nYear {year}:")
    
    categories = ['scaler', 'hgf', 'consistent_hgf', 'consistent_hypergrower', 
                  'gazelle', 'mature_hgf', 'scaleup', 'superstar']
    
    for cat in categories:
        col = f'{prefix}{cat}'
        if col in df.columns:
            count_1 = (df[col] == 1).sum()
            count_0 = (df[col] == 0).sum()
            count_na = (df[col] == 'n.a.').sum()
            print(f"  {cat:25s} | 1: {count_1:6d} | 0: {count_0:6d} | n.a.: {count_na:6d}")


=== Classification Summary ===

Year 2020:
  scaler                    | 1:    423 | 0:   3762 | n.a.:  41900
  hgf                       | 1:    129 | 0:   4056 | n.a.:  41900
  consistent_hgf            | 1:      0 | 0:   4185 | n.a.:  41900
  consistent_hypergrower    | 1:      0 | 0:   4185 | n.a.:  41900
  gazelle                   | 1:      0 | 0:      0 | n.a.:  46085
  mature_hgf                | 1:      0 | 0:      0 | n.a.:  46085
  scaleup                   | 1:      0 | 0:      0 | n.a.:  46085
  superstar                 | 1:      0 | 0:      0 | n.a.:  46085

Year 2021:
  scaler                    | 1:   1072 | 0:  11363 | n.a.:  33650
  hgf                       | 1:    313 | 0:  12122 | n.a.:  33650
  consistent_hgf            | 1:      0 | 0:  12435 | n.a.:  33650
  consistent_hypergrower    | 1:      0 | 0:  12435 | n.a.:  33650
  gazelle                   | 1:      0 | 0:      0 | n.a.:  46085
  mature_hgf                | 1:      0 | 0:      0 | n.a.:  46085
  scal

In [ ]:
# List all new columns
print(f"\nTotal columns created: {len(df.columns)}")
print(f"\nAll columns:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col}")

## 4. Save Extended Dataset

In [6]:
# Save the extended dataset
output_path = '../data/processed/austria_extended_classifications.pkl'
df.to_pickle(output_path)
print(f"✓ Extended dataset saved to: {output_path}")
print(f"  Shape: {df.shape}")
print(f"  Columns: {len(df.columns)}")
print(f"\nKeywords for variables:")
print(f"  - Growth variables: growth_YYYY (5 years × 3 years each)")
print(f"  - AAGR variables: aagr_YYYY (5 years)")
print(f"  - Classification variables: YYYY_[category] (5 years × 8 categories)")
print(f"  - Total new variables: ~50")

✓ Extended dataset saved to: ../data/processed/austria_extended_classifications.pkl
  Shape: (46085, 79)
  Columns: 79

Keywords for variables:
  - Growth variables: growth_YYYY (5 years × 3 years each)
  - AAGR variables: aagr_YYYY (5 years)
  - Classification variables: YYYY_[category] (5 years × 8 categories)
  - Total new variables: ~50
